## Generate Antenna Phase Center Offsets

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pylupnt as pnt
import datetime
import os
import pandas as pd

In [ ]:
# Load ray tracing pickele file
def get_outfile_name(sim_params, savedir, suffix="", format="pkl"):
    ymdh = sim_params["epoch_ymdh"]
    lcrns_idx = sim_params["lcrns_idx"]
    signal_family = sim_params["signal_family"]
    rz12 = sim_params["rz12"]
    gnss_const = sim_params["gnss_const"]
    kp = sim_params.get("kp", None)
    n_orbit = sim_params.get("n_orbit", 3)
    dt = sim_params.get("dt", 10)
    dtrt = sim_params.get("dt_raytrace", 60)

    orbit_dt_str = "norbit_{}_dt_{}s_dtrt_{}s".format(int(n_orbit), int(dt), int(dtrt))

    epoch_dict = {
        "year": ymdh[0],
        "month": ymdh[1],
        "day": ymdh[2],
        "hour": ymdh[3],
        "minute": 0,
        "second": 0,
    }
    epoch_str = (
        "{year}_{month:02d}_{day:02d}_{hour:02d}_{minute:02d}_{second:02d}".format(
            **epoch_dict
        )
    )

    outfilename = (
        "ionodata_raytrace_{}_sat_{}_{}_signal_{}_rz12_{:.1f}_kp_{:.1f}{}.{}".format(
            epoch_str, lcrns_idx, gnss_const, signal_family, rz12, kp, suffix, format
        )
    )

    savedir = os.path.join(savedir, orbit_dt_str)
    if os.path.exists(savedir) is False:
        os.makedirs(savedir)
    updated_pickle_filename = os.path.join(savedir, outfilename)

    return updated_pickle_filename


def get_pco_fielename(sim_params, savedir, format="pkl"):
    ymdh = sim_params["epoch_ymdh"]
    lcrns_idx = sim_params["lcrns_idx"]
    signal_family = sim_params["signal_family"]
    rz12 = sim_params["rz12"]
    gnss_const = sim_params["gnss_const"]
    kp = sim_params.get("kp", None)
    n_orbit = sim_params.get("n_orbit", 3)
    dt = sim_params.get("dt", 10)
    dtrt = sim_params.get("dt_raytrace", 60)

    orbit_dt_str = "norbit_{}_dt_{}s_dtrt_{}s".format(int(n_orbit), int(dt), int(dtrt))

    epoch_dict = {
        "year": ymdh[0],
        "month": ymdh[1],
        "day": ymdh[2],
        "hour": ymdh[3],
        "minute": 0,
        "second": 0,
    }
    epoch_str = (
        "{year}_{month:02d}_{day:02d}_{hour:02d}_{minute:02d}_{second:02d}".format(
            **epoch_dict
        )
    )

    outfilename = "antenna_pco_{}_sat_{}_{}_signal_{}.{}".format(
        epoch_str, lcrns_idx, gnss_const, signal_family, format
    )

    savedir = os.path.join(savedir, orbit_dt_str)
    if os.path.exists(savedir) is False:
        os.makedirs(savedir)
    updated_pickle_filename = os.path.join(savedir, outfilename)

    return updated_pickle_filename


def load_pickle_file(sim_params, savedir):
    updated_pickle_filename = get_outfile_name(sim_params, savedir, format="pkl")

    if os.path.exists(updated_pickle_filename):
        print(f"Loading {updated_pickle_filename}")
        df = pd.read_pickle(updated_pickle_filename)
    else:
        print(f"{updated_pickle_filename} not found.")
        df = None

    return df

In [ ]:
# Load L1, L5 signal for each constellation
savedir = os.path.join(pnt.get_output_dir(), "iono_delay", "raytrace_summary")
gnss_consts = ["GPS", "GALILEO", "QZSS"]

df_signals = {}
for gnss_const in gnss_consts:
    df_signals[gnss_const] = {}
    for signal_family in [1, 5]:
        sim_params = {
            "gnss_const": gnss_const,
            "signal_family": signal_family,
            "lcrns_idx": 0,
            "epoch_ymdh": [2025, 3, 1, 12],
            "rz12": 50.0,  # -1.0 for historical or projected R12
            "kp": 3.0,
            "n_orbit": 6,
            "dt": 1,  # time step in seconds
            "dt_raytrace": 120,  # raytracing time step in seconds
        }
        df_signals[gnss_const][signal_family] = load_pickle_file(sim_params, savedir)

In [ ]:
df_head = df_signals["GPS"][1].head()

In [ ]:
def get_gnss_str(gnss_const):
    if gnss_const == "GPS":
        return "G"
    elif gnss_const == "GALILEO":
        return "E"
    elif gnss_const == "QZSS":
        return "J"
    else:
        raise ValueError(f"Unknown GNSS constellation: {gnss_const}")


def get_freq_str(gnss_const, signal_family):
    if gnss_const in ["GPS", "QZSS"]:
        if signal_family == 1:
            return "L1"
        elif signal_family == 2:
            return "L2"
        elif signal_family == 5:
            return "L5"
    elif gnss_const == "GALILEO":
        if signal_family == 1:
            return "E1"
        elif signal_family == 5:
            return "E5a"
    raise ValueError(
        f"Unknown frequency for {gnss_const} signal family {signal_family}"
    )


def unit(v, eps=1e-12):
    norm = np.linalg.norm(v)
    if norm < eps:
        raise ValueError("Cannot normalize zero-length vector.")
    return v / norm


def enu_to_ecef_rot(t_tai, r_sat_ecef):
    r = np.asarray(r_sat_ecef, dtype=float).reshape(3)

    # Up: radial outward (geocentric up)
    u = unit(r, eps=1e-12)

    # Earth spin axis in ECEF (approx): z-axis
    z = np.array([0.0, 0.0, 1.0])

    # East: tangent direction
    # e = z x u (points east unless near poles)
    e_raw = np.cross(z, u)
    if np.linalg.norm(e_raw) < 1e-12:
        # Satellite nearly over pole; choose a different reference axis
        # Use x-axis as fallback
        x = np.array([1.0, 0.0, 0.0])
        e_raw = np.cross(x, u)
        if np.linalg.norm(e_raw) < 1e-12:
            raise ValueError("Cannot define East vector (degenerate geometry).")
    e = unit(e_raw, eps=1e-12)

    # North: complete right-handed frame (N = U x E)
    n = unit(np.cross(u, e), eps=1e-12)
    Cneu = np.column_stack((n, e, u))  # columns are NEU axes in ECEF

    return Cneu


def ijk_to_ecef_rot(t_tai, r_sat_ecef):
    r = np.asarray(r_sat_ecef, dtype=float).reshape(3)

    # z-axis: radial outward (geocentric up)
    kvec = -unit(r, eps=1e-12)

    # x-axis: projection of ECEF x-axis onto plane perpendicular to z
    r_sun = pnt.get_body_pos_vel(t_tai, pnt.EARTH, pnt.SUN, pnt.ECEF)[:3]
    jvec = unit(r_sun - r_sat_ecef, eps=1e-12)  # approximate Earth-Sun direction

    ivec = np.cross(jvec, kvec)

    Cijk = np.column_stack((ivec, jvec, kvec))  # columns are IJK axes in ECEF

    return Cijk

In [ ]:
from tqdm import tqdm

atx = pnt.get_file_path("igs20.atx")
loader = pnt.ANTEXLoader(atx)
gnss_consts = ["GPS", "GALILEO", "QZSS"]

fig, axes = plt.subplots(len(gnss_consts), 2, figsize=(12, 4 * len(gnss_consts)))

for i, gnss_const in enumerate(gnss_consts):
    gnss_str = get_gnss_str(gnss_const)
    for j, signal_family in enumerate([1, 5]):
        freq_str = get_freq_str(gnss_const, signal_family)
        df = df_signals[gnss_const][signal_family]
        t_tai = df["t_tai"].values
        tspan = df["tspan"].values
        pos_tx_sp3 = df["pos_tx"].values * 1000  # ECEF position of transmitter
        pos_tx_ephem = df["pos_tx_ephem"].values * 1000  # ECEF position from ephemeris
        print(pos_tx_sp3.shape)
        prn = df["prn"].values
        lent = len(t_tai)
        pco_neu = np.zeros((lent, 3))
        pco_ecef = np.zeros((lent, 3))
        diff_ephem_pco = np.zeros(lent)
        diff_ephem_raw = np.zeros(lent)

        # clock median across constellation
        clock_const = np.median(
            df["clockbias_ephem"].values - df["clockbias_tx"].values
        )
        df_signals[gnss_const][signal_family]["clock_bias_tx_median"] = (
            clock_const * np.ones(pco_ecef.shape[0])
        )

        print(f"Processing {gnss_const} {freq_str}:")
        print("  Median Clock Bias (m): ", clock_const * pnt.C)

        # run if "pco_ecef_x_m" not in df_signals[gnss_const][signal_family].columns:
        if "pco_ecef_x_m" not in df_signals[gnss_const][signal_family].columns:
            for k in tqdm(range(lent), desc=f"Processing {gnss_const} {freq_str} PCO"):
                dt = pnt.tai_to_datetime(t_tai[k])
                try:
                    pco_neu[k] = loader.get_pco(
                        dt, gnss_str, prn[k], freq=freq_str, freqcode=None
                    )
                except Exception as e:
                    # Use zero PCO if not found
                    pco_neu[k] = np.zeros(3)

                rotmat = ijk_to_ecef_rot(t_tai[k], pos_tx_sp3[k])
                pco_ecef[k] = rotmat @ pco_neu[k]

                diff_ephem_pco[k] = np.linalg.norm(
                    (pos_tx_sp3[k] + pco_ecef[k]) - (pos_tx_ephem[k] + np.zeros(3))
                )
                diff_ephem_raw[k] = np.linalg.norm(pos_tx_sp3[k] - pos_tx_ephem[k])
        else:
            print(
                f"PCO ECEF data already exists for {gnss_const} {freq_str}, skipping computation."
            )
            pco_ecef[:, 0] = df_signals[gnss_const][signal_family][
                "pco_ecef_x_m"
            ].values
            pco_ecef[:, 1] = df_signals[gnss_const][signal_family][
                "pco_ecef_y_m"
            ].values
            pco_ecef[:, 2] = df_signals[gnss_const][signal_family][
                "pco_ecef_z_m"
            ].values
            for k in range(lent):
                diff_ephem_pco[k] = np.linalg.norm(
                    (pos_tx_sp3[k] + pco_ecef[k]) - (pos_tx_ephem[k] + np.zeros(3))
                )
                diff_ephem_raw[k] = np.linalg.norm(pos_tx_sp3[k] - pos_tx_ephem[k])

        if gnss_const == "QZSS":
            unique_prns = np.unique(prn)
            print(f"  Unique PRNs: {unique_prns}")
            for prn_val in unique_prns:
                prn_mask = prn == prn_val
                pos_diff_ephem_pco = np.zeros((np.sum(prn_mask), 3))
                jj = 0
                for k in range(lent):
                    if prn_mask[k]:
                        pos_diff_ephem_pco[jj, :] = (
                            pos_tx_sp3[k] + pco_ecef[k] - pos_tx_ephem[k]
                        )
                        jj += 1
                        diff_ephem_pco[k] = np.linalg.norm(
                            pos_diff_ephem_pco[jj - 1, :]
                        )
                med_diff_ephem_pco = np.median(pos_diff_ephem_pco, axis=0)
                # offset pco_ecef for this PRN by subtracting the median
                pco_ecef[prn_mask] -= med_diff_ephem_pco

        df_signals[gnss_const][signal_family]["pco_ecef_x_m"] = pco_ecef[:, 0]
        df_signals[gnss_const][signal_family]["pco_ecef_y_m"] = pco_ecef[:, 1]
        df_signals[gnss_const][signal_family]["pco_ecef_z_m"] = pco_ecef[:, 2]

        print(
            "  90th percentile difference with PCO (m): ",
            np.percentile(diff_ephem_pco, 90),
        )
        print(
            "  90th percentile difference raw ephem (m): ",
            np.percentile(diff_ephem_raw, 90),
        )

        # Plot tspan vs PCO ECEF x, y, z
        # Plot histogram of differences
        bins = np.linspace(0, 5, 100)
        if len(gnss_consts) == 1:
            ax = axes[j]
        else:
            ax = axes[i, j]

        ax.set_title(f"{gnss_const} {freq_str} Ephem Position Differences")
        ax.hist(diff_ephem_pco, bins=bins, alpha=0.5, label="With PCO")
        ax.hist(diff_ephem_raw, bins=bins, alpha=0.5, label="Raw Ephem")
        ax.set_xlabel("Differences (m)")
        ax.grid(True)
        ax.legend()

plt.tight_layout()
plt.show()

## Save to the File  with Phase Center Offset

In [ ]:
# Save the updated dataframes with PCO information
for gnss_const in gnss_consts:
    for signal_family in [1, 5]:
        sim_params = {
            "gnss_const": gnss_const,
            "signal_family": signal_family,
            "lcrns_idx": 0,
            "epoch_ymdh": [2025, 3, 1, 12],
            "rz12": 50.0,  # -1.0 for historical or projected R12
            "kp": 3.0,
            "n_orbit": 6,
            "dt": 1,  # time step in seconds
            "dt_raytrace": 120,  # raytracing time step in seconds
        }

        # Save updated dataframe with PCO
        tidx = df_signals[gnss_const][signal_family]["tidx"].values
        df_pco_x = df_signals[gnss_const][signal_family]["pco_ecef_x_m"].values
        df_pco_y = df_signals[gnss_const][signal_family]["pco_ecef_y_m"].values
        df_pco_z = df_signals[gnss_const][signal_family]["pco_ecef_z_m"].values

        # create a dataframe with only df_pco_x, df_pco_y, df_pco_z and no index
        df = pd.DataFrame(
            columns=["tidx", "pco_ecef_x_m", "pco_ecef_y_m", "pco_ecef_z_m"],
            index=range(len(tidx)),
        )
        df["tidx"] = tidx
        df["pco_ecef_x_m"] = df_pco_x
        df["pco_ecef_y_m"] = df_pco_y
        df["pco_ecef_z_m"] = df_pco_z

        print(df.head())

        savedir = os.path.join(pnt.get_output_dir(), "iono_delay", "antenna_offset")
        savefile = get_pco_fielename(sim_params, savedir, format="pkl")
        print(f"Saving PCO to {savefile}")
        df.to_pickle(savefile)

        # save to csv
        savefile_csv = savefile.replace(".pkl", ".csv")
        print(f"Saving PCO to {savefile_csv}")
        df.to_csv(savefile_csv, index=False)

## Saving PCO and Rest of Files into a same file

In [ ]:
columns = df_head.columns.tolist() + [
    "clock_bias_tx_median",
    "pco_ecef_x_m",
    "pco_ecef_y_m",
    "pco_ecef_z_m",
]
print(columns)

In [ ]:
print(df_signals["GPS"][1]["pco_ecef_x_m"].values)

In [ ]:
# Save the updated dataframes with PCO information
gnss_consts = ["GPS", "GALILEO", "QZSS"]

for gnss_const in gnss_consts:
    for signal_family in [1, 5]:
        sim_params = {
            "gnss_const": gnss_const,
            "signal_family": signal_family,
            "lcrns_idx": 0,
            "epoch_ymdh": [2025, 3, 1, 12],
            "rz12": 50.0,  # -1.0 for historical or projected R12
            "kp": 3.0,
            "n_orbit": 6,
            "dt": 1,  # time step in seconds
            "dt_raytrace": 120,  # raytracing time step in seconds
        }

        # copy dataframe
        df = pd.DataFrame(
            columns=columns, index=df_signals[gnss_const][signal_family].index
        )
        for col in columns:
            df[col] = df_signals[gnss_const][signal_family][col].values

        print(df.head())

        savedir = os.path.join(
            pnt.get_output_dir(), "iono_delay", "raytrace_summary_pco"
        )
        savefile = get_outfile_name(sim_params, savedir, suffix="_pco", format="pkl")
        print(f"Saving updated dataframe with PCO to {savefile}")
        df.to_pickle(savefile)

        # save to csv
        savefile_csv = savefile.replace(".pkl", ".csv")
        print(f"Saving updated dataframe with PCO to {savefile_csv}")
        df.to_csv(savefile_csv, index=False)